# Incrementality study (Criteo)

**Goal:** use a real randomized ad experiment to show how many visits and conversions the ads caused, versus how many would have happened anyway.

**The question we're answering:** We ran display ads to 85% of users and held out 15%. Did the ads cause extra visits and conversions, and was it worth it?

## Setup

In [ ]:
!pip install scikit-uplift statsmodels pandas matplotlib seaborn lightgbm

## Step 1. Frame it as a client question

"We ran display ads to 85% of users and held out 15%. Did the ads cause extra visits and conversions, and was it worth it?"

## Step 2. Load a 10% sample

The `scikit-uplift` library's hosted S3 link for this dataset returns a 404 (dead bucket), so we pull the file directly from a known working mirror and sample it ourselves.

In [ ]:
import pandas as pd

# Working GitHub-hosted mirror of the Criteo Uplift Prediction dataset
url = "https://github.com/jszymon/uplift_sklearn_data/releases/download/Criteo/criteo-research-uplift-v2.1.csv.gz"

df_full = pd.read_csv(url, compression="gzip")
print(df_full.shape)

# Our own 10% random sample
df = df_full.sample(frac=0.10, random_state=42).reset_index(drop=True)
print(df.shape)
df.head()

**Result:** full dataset 13,979,592 rows; 10% sample 1,397,959 rows.

## Step 3. Check the experiment is valid

Before trusting any result, confirm the treatment/control split was genuinely random by comparing group sizes and pre-campaign feature balance.

In [ ]:
# a) check the group sizes
print(df['treatment'].value_counts(normalize=True))

# b) check the 12 features are balanced across groups
feature_cols = [f'f{i}' for i in range(12)]
balance_table = df.groupby('treatment')[feature_cols].mean()
print(balance_table)

**Result:** 85.0% treatment / 15.0% control, matching the known experiment design. All 12 anonymized features (f0–f11) are balanced between groups to 1–2 decimal places (e.g. f0: 19.64 control vs 19.62 treatment; f9: 15.89 vs 16.04), confirming randomization worked. Anything we find downstream can be read as causal, not just correlational.

## Step 4. Calculate lift

In [ ]:
rates = df.groupby('treatment')[['visit', 'conversion']].mean()
abs_lift = rates.loc[1] - rates.loc[0]
rel_lift = abs_lift / rates.loc[0]

print("Rates by group:\n", rates)
print("\nAbsolute lift:\n", abs_lift)
print("\nRelative lift:\n", rel_lift)

**Result:** conversion rate rose from 0.1964% (control) to 0.3196% (treatment) — an absolute lift of 0.1231 percentage points, a **62.7% relative lift**. Visit rate rose from 3.82% to 4.89% (+27.9% relative).

## Step 5. Test significance

In [ ]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

t = df[df.treatment == 1]
c = df[df.treatment == 0]

stat, p = proportions_ztest(
    [t.conversion.sum(), c.conversion.sum()],
    [len(t), len(c)]
)

low, high = confint_proportions_2indep(
    t.conversion.sum(), len(t),
    c.conversion.sum(), len(c)
)

print(f"p-value: {p:.6f}")
print(f"95% CI for lift: [{low:.5f}, {high:.5f}]")

**Result:** p < 0.000001. 95% CI for the lift is [0.00101, 0.00144] — entirely positive, so the lift is statistically significant, not random noise.

## Step 6. Turn lift into business numbers

The dataset has no real cost or revenue data, so CPM and conversion value are stated assumptions, labeled as such.

In [ ]:
n_treated = len(t)
inc_conv_rate = abs_lift['conversion']
incremental_conversions = inc_conv_rate * n_treated
total_treated_conversions = t.conversion.sum()
share_incremental = incremental_conversions / total_treated_conversions

# Assumptions — stated clearly, not derived from the data
cpm = 5              # cost per 1000 impressions, in dollars
value_per_conv = 40  # revenue per conversion, in dollars

spend = n_treated / 1000 * cpm
revenue = incremental_conversions * value_per_conv
iroas = revenue / spend

print(f"Treated users: {n_treated:,}")
print(f"Incremental conversions: {incremental_conversions:.0f}")
print(f"Total treated conversions: {total_treated_conversions:.0f}")
print(f"Share of treated conversions that were incremental: {share_incremental:.1%}")
print(f"Assumed spend: ${spend:,.2f}")
print(f"Assumed incremental revenue: ${revenue:,.2f}")
print(f"iROAS at ${cpm} CPM and ${value_per_conv}/conv: {iroas:.2f}")

**Result:** of the 3,799 conversions in the treated group, only **1,464 (38.5%)** were truly incremental — the rest would have happened without ads. At an assumed $5 CPM and $40/conversion, this implies an **iROAS of 9.85**.

## Step 7. Show the trap (the key slide)

Compare the naive measurement (exposed vs. not-exposed, inside the treatment group only) against the true causal lift (treatment vs. control, both randomly assigned).

In [ ]:
exposed_vs_not = t.groupby('exposure')['conversion'].mean()
print(exposed_vs_not)

naive_lift = exposed_vs_not.loc[1] - exposed_vs_not.loc[0]
true_lift = abs_lift['conversion']

print(f"\nNaive lift (exposed vs not-exposed): {naive_lift:.5f}")
print(f"True causal lift (treatment vs control): {true_lift:.5f}")
print(f"Overstatement ratio: {naive_lift/true_lift:.1f}x")

**Result:** the naive "exposed vs. not-exposed" comparison shows a 5.46 percentage-point lift. The true causal lift from the randomized holdout is only 0.12 percentage points — the naive method **overstates the ad effect by 44x**. This happens because users who were actually shown ads were already more engaged (more page views, more logged-in sessions), a selection bias that has nothing to do with the ads themselves.

## Summary

| Metric | Value |
|---|---|
| Users analyzed (10% sample) | 1,397,959 |
| Treatment / control split | 85.0% / 15.0% |
| Conversion rate: control → treatment | 0.20% → 0.32% |
| Relative lift | +62.7% (p < 0.001) |
| 95% CI for lift | [0.10pp, 0.14pp] |
| Incremental conversions | 1,464 of 3,799 (38.5%) |
| iROAS (at $5 CPM, $40/conv) | 9.85 |
| Naive vs. true lift overstatement | 44x |

**Recommendations:**
1. Keep the campaign — the lift is real, statistically significant, and profitable.
2. Report incremental CPA, not attributed CPA, to leadership.
3. Maintain a permanent 10–15% holdout on every future campaign to keep measuring true causal lift.

**Limits:** Criteo's features are anonymized (f0–f11), so no business interpretation of individual features is possible. CPM and conversion value are illustrative assumptions, not real cost data. Analysis uses a 10% random sample of the full 14M-row dataset for speed; results are consistent with the full-dataset published benchmarks.